# King County house sales: EDA for Zachary Brooks

**Client:** Zachary Brooks, seller. Invests in historical houses and wants to sell in the best neighbourhoods, at the best time of year, for high profits. He also wants to know whether renovating is worthwhile.

**Business question:** Where can the client sell older houses in good neighbourhoods for large returns, and when should he sell?
**Secondary question:** Are older houses worth renovating?

## Hypotheses

- **H1 (age and location):** Older (pre-1940) houses sell for a lower price per sqft than newer houses, and some zip codes command a significantly higher price per sqft than others.
- **H2 (renovation):** Renovated older houses sell for more than unrenovated ones.
- **H3 (timing):** Sale prices of older houses vary by month or season, with spring higher than winter.

## Data

King County (Seattle area) house sales from May 2014 to May 2015: 21,597 sales of 21,420 houses. The data was extracted from the course database with `fetch_data.py` and saved to `data/eda.csv`, so this notebook works from the local CSV and doesn't need a database connection.

**Key definitions used throughout:**
- **Older house:** built before 1940 (pre-war housing stock).
- **Price per sqft (ppsf):** sale price ÷ living area, used to compare houses of different sizes fairly.
- **Good neighbourhood:** a zip code in the top quartile of median price per sqft.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
import contextily as cx
from matplotlib.patches import Patch

# One plot style for the whole notebook
sns.set_theme(style="white")

# Load the extracted data
df = pd.read_csv("data/eda.csv")

## 2. Data overview and cleaning

`df.head()` gives a first look at the data. Two issues are already visible: `yr_renovated` contains values like 19910, and both `NaN` and `0` appear in `yr_renovated` and `waterfront`.

In [ ]:
df.head(10)

`df.info()` shows the column types and where values are missing. The column names are already lower case without spaces, so they don't need cleaning. There are no duplicate rows. `date` is stored as text, so it's converted to a datetime.

In [ ]:
df.info()
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Convert "date" from text to datetime
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
print(f"date dtype: {df['date'].dtype}")

### 2.1 Fixing `yr_renovated`

All non-zero renovation years are between 19,340 and 20,150 and end in 0, so every one of them has been stored ×10 (e.g. 19910 = 1991). After dividing by 10, no house has a renovation year earlier than its build year, which confirms the fix.

In [ ]:
# Non-missing, non-zero renovation years
renovated_years = df.loc[df["yr_renovated"].notna() & (df["yr_renovated"] != 0), "yr_renovated"]
print(f"Range: {renovated_years.min():.0f} to {renovated_years.max():.0f}")
print(f"All end in 0: {(renovated_years % 10 == 0).all()}")

# Check: renovated before built after the fix?
fixed = df["yr_renovated"] / 10
print(f"Renovated before built: {(fixed[fixed > 0] < df.loc[fixed > 0, 'yr_built']).sum()}")

# Apply the fix. The if-check prevents dividing again if the cell is re-run.
if df["yr_renovated"].max() > 3000:
    df["yr_renovated"] = df["yr_renovated"] / 10

### 2.2 Missing renovation years: `NaN` vs `0`

To decide how to treat missing renovation years, the `NaN` rows are compared with the `0` rows and with the renovated houses on median year built, median price per sqft, average condition and average grade.

The `NaN` rows are almost identical to the `0` rows and clearly different from the renovated ones, so both are treated as **"not renovated"**. A Boolean column `renovated` records this. Some of these houses may have been renovated without it being recorded, which is a limitation of the data.

In [ ]:
# Price per square foot, used throughout for fairer comparison between houses of different sizes
df["price_per_sqft"] = df["price"] / df["sqft_living"]

# Label each row as NaN, 0 or Renovated
renovation_status = np.select(
    [df["yr_renovated"].isna(), df["yr_renovated"] == 0],
    ["NaN", "0"],
    default="Renovated",
)

comparison = df.groupby(renovation_status).agg(
    count=("price", "size"),
    median_yr_built=("yr_built", "median"),
    median_price_per_sqft=("price_per_sqft", "median"),
    avg_condition=("condition", "mean"),
    avg_grade=("grade", "mean"),
).reindex(["NaN", "0", "Renovated"])
comparison.index.name = "renovation_status"

display(comparison.style.format({
    "count": "{:,}",
    "median_yr_built": "{:.0f}",
    "median_price_per_sqft": "${:,.0f}",
}, precision=2))

# NaN and 0 both become False
df["renovated"] = df["yr_renovated"] > 0

### 2.3 Helper columns

- `zipcode` is converted to text so it's treated as a category, not a number.
- `is_old` marks houses built before 1940.

In [ ]:
df["zipcode"] = df["zipcode"].astype(str)
df["is_old"] = df["yr_built"] < 1940

print(f"Zip codes: {df['zipcode'].nunique()}")
print(f"Pre-1940 sales: {df['is_old'].sum():,} ({df['is_old'].mean():.1%} of all sales)")

## 3. Price distribution

Sale prices are heavily right-skewed: the upper quartile starts at about \$645k and runs to \$7.7M. The left histogram is cut off at \$2M for readability, which leaves out less than 1% of sales.

On a log scale the price distribution looks close to a normal distribution. Because of the skew, **medians** are used instead of means throughout, as they're far less affected by a few very expensive houses. A log scale also reflects how prices actually behave: a \$50k difference matters a lot on a \$300k house and much less on a \$3M house.

In [ ]:
print(f"Sales above $2M: {(df['price'] > 2_000_000).sum()} ({(df['price'] > 2_000_000).mean():.2%})")

with pd.option_context("display.float_format", "{:,.2f}".format):
    display(df[["price", "price_per_sqft"]].describe().T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["price"], bins=100, color="g", alpha=0.4, kde=True, ax=axes[0])
axes[0].set_title("Sale price (axis cut at $2M)")
axes[0].set_xlabel("Price (USD)")
axes[0].set_ylabel("Number of sales")
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
axes[0].set_xlim(0, 2_000_000)

sns.histplot(df["price_per_sqft"], bins=50, color="b", alpha=0.4, kde=True, ax=axes[1])
axes[1].set_title("Price per square foot")
axes[1].set_xlabel("Price per sqft (USD)")
axes[1].set_ylabel("Number of sales")
axes[1].xaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.histplot(df["price"], bins=100, log_scale=True, color="g", alpha=0.4, kde=True, ax=ax)
ax.set_title("Distribution of sale prices (log scale)")
ax.set_xlabel("Sale price (USD, log scale)")
ax.set_ylabel("Number of sales")

ax.set_xticks([100_000, 200_000, 500_000, 1_000_000, 2_000_000, 5_000_000])
ax.xaxis.set_major_formatter(mtick.FuncFormatter(
    lambda x, _: f"${x/1e6:.0f}M" if x >= 1e6 else f"${x/1e3:.0f}k"
))
ax.xaxis.set_minor_formatter(mtick.NullFormatter())   # hide extra minor-tick labels

sns.despine()
plt.show()

## 4. What drives price and price per sqft?

Spearman correlation is used because it suits skewed data and isn't distorted by extreme values.

In [ ]:
cols = ["price", "price_per_sqft", "sqft_living", "grade", "condition",
        "yr_built", "lat", "long", "view", "waterfront", "bedrooms", "bathrooms"]

corr = df[cols].corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool))   # show each pair only once

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Spearman correlations")
plt.tight_layout()
plt.show()

- **Grade** has the strongest correlation with price but only a weak one with price per sqft. High-grade houses tend to be larger, so most of grade's link to price comes through size.
- **Latitude** is the only variable that is strong for both. Houses further north sell not just for higher prices, but for higher prices *for their size*. Location should therefore be a key part of the analysis.
- **Year built** is negatively correlated with price per sqft but positively with price and living area. Older houses tend to be smaller and cheaper in total, but sell for more per square foot. This may partly be location, since older houses cluster in the north.
- **Condition** is weak for both measures across the whole dataset. That's worth keeping in mind for renovation, though it could change once location is controlled for.

## 5. H1: Age and location

### 5.1 Good neighbourhoods

There are 70 zip codes. The top quartile by median price per sqft (18 zip codes, cutoff ≈ \$313/sqft) is defined as **"good neighbourhoods"**. Median price per sqft varies widely between zip codes, which supports the location part of H1.

In [ ]:
zip_comparison = df.groupby("zipcode").agg(
    median_price_per_sqft=("price_per_sqft", "median"),
    no_of_sales=("price", "size"),
    no_of_old_sales=("is_old", "sum"),
    median_price=("price", "median"),
).sort_values("median_price_per_sqft", ascending=False)

# Top quartile of zip codes = "good neighbourhoods"
cutoff = zip_comparison["median_price_per_sqft"].quantile(0.75)
top_quartile = zip_comparison[zip_comparison["median_price_per_sqft"] >= cutoff]

print(f"Cutoff: ${cutoff:,.0f} per sqft, {len(top_quartile)} zip codes")
display(top_quartile.style.format({
    "no_of_sales": "{:,}",
    "no_of_old_sales": "{:,}",
    "median_price_per_sqft": "${:,.0f}",
    "median_price": "${:,.0f}",
}))

df["good_neighbourhood"] = df["zipcode"].isin(top_quartile.index)

### 5.2 Older vs newer houses within zip codes

Older and newer houses are compared within each zip code that has **at least 20 older-house sales**; fewer than that gives unreliable medians. That leaves 30 zip codes.

Across these, older houses sell for a median of **8.8% more per sqft** than newer houses in the same zip code, but for about **0.7% less in total price**, because they tend to be smaller. The first part of H1 is therefore **not supported**: pre-1940 houses are worth *more* per square foot, not less, even within the same location.

In [ ]:
by_zip = df.pivot_table(index="zipcode", columns="is_old",
                        values=["price_per_sqft", "price"], aggfunc="median")

# Flatten the two-level column names, e.g. ("price", True) -> "older_price"
by_zip.columns = [
    f"{'older' if is_old else 'newer'}_{'ppsf' if measure == 'price_per_sqft' else 'price'}"
    for measure, is_old in by_zip.columns
]

by_zip["n_older"] = df[df["is_old"]].groupby("zipcode").size()
by_zip["older_ppsf_premium_%"] = (by_zip["older_ppsf"] / by_zip["newer_ppsf"] - 1) * 100
by_zip["older_price_premium_%"] = (by_zip["older_price"] / by_zip["newer_price"] - 1) * 100

by_zip = by_zip[by_zip["n_older"] >= 20].sort_values("older_ppsf", ascending=False)

ppsf_prem = by_zip["older_ppsf_premium_%"].median()
price_prem = by_zip["older_price_premium_%"].median()
print(f"Median older-house premium: {ppsf_prem:.1f}% per sqft, {price_prem:.1f}% on total price "
      f"({len(by_zip)} zip codes with at least 20 older-house sales)")

by_zip.head(10).round(1)

### 5.3 Best good neighbourhoods for older houses

The charts show the top good-neighbourhood zip codes for older houses by price per sqft and by total sale price. Zip codes in the top 10 on **both** measures are highlighted; these are the strongest candidates for the client.

In [ ]:
good = by_zip[by_zip.index.isin(top_quartile.index)]

good_old_ppsf  = good.sort_values("older_ppsf",  ascending=False).head(10)
good_old_price = good.sort_values("older_price", ascending=False).head(10)

in_both = set(good_old_ppsf.index) & set(good_old_price.index)
highlight, base = "darkorange", "cornflowerblue"

fig, axes = plt.subplots(1, 2, figsize=(16, 8))   # 1 row, 2 columns
fig.suptitle("Top 'Good Neighbourhood' zip codes for pre-1940 houses", fontsize=16, fontweight="bold")

sns.barplot(x=good_old_ppsf["older_ppsf"], y=good_old_ppsf.index, order=good_old_ppsf.index,
            color="cornflowerblue", ax=axes[0])

axes[0].set_title("By median price per sqft")
axes[0].set_xlabel("Median price per sqft (USD)")
axes[0].set_ylabel("Zipcode")
axes[0].xaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))

sns.barplot(x=good_old_price["older_price"], y=good_old_price.index, order=good_old_price.index,
            color="cornflowerblue", ax=axes[1])

axes[1].set_title("By median sale price")
axes[1].set_xlabel("Median price (USD)")
axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))

for ax, table in [(axes[0], good_old_ppsf), (axes[1], good_old_price)]:
    for bar, zipcode in zip(ax.patches, table.index):
        bar.set_color(highlight if zipcode in in_both else base)
    for label in ax.get_yticklabels():
        if label.get_text() in in_both:
            label.set_fontweight("bold")

fig.legend(handles=[Patch(color=highlight, label="In top 10 for both measures"),
                    Patch(color=base, label="In one list only")],
           loc="lower center", ncol=2, frameon=False)

sns.despine()

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

print(f"In the top 10 for both measures: {', '.join(sorted(in_both))}")

### 5.4 Where are the older houses?

The maps show all sales and then the pre-1940 houses, coloured by price per sqft. Older houses cluster in central and north Seattle, around the western shore of Lake Washington, which is also where prices per sqft are highest. The maps are zoomed in on central Seattle and still include 95% of older-house sales.

In [ ]:
vmin, vmax = df["price_per_sqft"].quantile([0.05, 0.95])
old = df[df["is_old"]]

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)

# Left: every sale, coloured by price per sqft
sc = axes[0].scatter(df["long"], df["lat"], c=df["price_per_sqft"], cmap="viridis",
                     vmin=vmin, vmax=vmax, s=3, alpha=0.6)
axes[0].set_title("All sales")

# Right: all sales in grey, with pre-1940 houses coloured on top
axes[1].scatter(df["long"], df["lat"], color="lightgrey", s=3, alpha=0.4)
axes[1].scatter(old["long"], old["lat"], c=old["price_per_sqft"], cmap="viridis",
                vmin=vmin, vmax=vmax, s=6, alpha=0.4)
axes[1].set_title("Pre-1940 houses (grey: all other sales)")

for ax in axes:
    ax.set_aspect(1 / np.cos(np.radians(47.5)))   # correct proportions at Seattle's latitude
    ax.set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

for ax in axes:
    ax.set_xlim(-122.44, -122.)   # longitude: west, east
    ax.set_ylim(47.35, 47.78)       # latitude: south, north
    cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.Esri.WorldGrayCanvas)

cbar = fig.colorbar(sc, ax=axes, shrink=0.7, label="Price per sqft (USD)")
cbar.ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
fig.suptitle("Price per square foot across King County", fontsize=16, fontweight="bold")
plt.show()

### 5.5 Spread of prices within the top zip codes

The box plots show the spread of price per sqft for older houses in the top zip codes, with each sale as a dot. A wide box means less predictable sale prices; a narrow one means more consistent prices.

In [ ]:
top_zips = good_old_ppsf.index
old_top = df[df["is_old"] & df["zipcode"].isin(top_zips)]

fig, ax = plt.subplots(figsize=(10, 7))
sns.boxplot(data=old_top, x="price_per_sqft", y="zipcode", order=top_zips,
            color="cornflowerblue", showfliers=False, width=0.6, ax=ax)
sns.stripplot(data=old_top, x="price_per_sqft", y="zipcode", order=top_zips,
              color="black", size=3, alpha=0.4, jitter=0.2, ax=ax)

ax.set_title("Price per sqft of pre-1940 houses in the top 'Good Neighbourhood' zip codes")
ax.set_xlabel("Price per sqft (USD)")
ax.set_ylabel("Zip code")
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
sns.despine()
plt.tight_layout()
plt.show()

## 6. H2: Renovation

### 6.1 First comparison

Renovated pre-1940 houses sell for a *lower* median price per sqft than unrenovated ones, but for a *higher* median sale price. They're also about 30% larger, which suggests renovations often add living space. Because larger houses have a lower price per sqft, price per sqft understates the effect of renovating here.

Comparing houses of similar size (the size-band table below), renovated houses still seem to sell for slightly less. That changes once location is controlled for as well (next section).

In [ ]:
ren_comparison = df.groupby(["is_old", "renovated"]).agg(
    no_of_sales=("price", "size"),
    median_price_per_sqft=("price_per_sqft", "median"),
    median_price=("price", "median"),
    median_sqft=("sqft_living", "median"),
    median_yr_built=("yr_built", "median"),
    avg_condition=("condition", "mean"),
    avg_grade=("grade", "mean"),
)

# Readable labels
ren_comparison = ren_comparison.rename(
    index={True: "Pre-1940", False: "1940+"}, level="is_old"
).rename(
    index={True: "Renovated", False: "Not renovated"}, level="renovated"
).rename_axis(["Built", "Renovation status"])

display(ren_comparison.style.format({
    "no_of_sales": "{:,}",
    "median_price_per_sqft": "${:,.0f}",
    "median_price": "${:,.0f}",
    "median_sqft": "{:,.0f}",
    "median_yr_built": "{:.0f}",
}, precision=2))

In [ ]:
# Older houses only, grouped into size bands
old = df[df["is_old"]].copy()
old["size_band"] = pd.cut(old["sqft_living"], bins=[0, 1200, 1800, 2500, 10_000],
                          labels=["<1,200", "1,200–1,800", "1,800–2,500", "2,500+"])

size_bands = old.pivot_table(index="size_band", columns="renovated", values="price",
                             aggfunc="median", observed=True)
size_bands.columns = ["Not renovated", "Renovated"]
size_bands.style.format("${:,.0f}")

### 6.2 Controlling for size *and* location

Renovated older houses are more common in cheaper zip codes, which pulls their overall median down. Comparing renovated and unrenovated houses within the same zip code **and** size band removes this effect. This is an example of **Simpson's paradox**: a trend that appears in the combined data reverses once the groups are separated.

Only **recent renovations (1990 or later)** are counted here, as a house renovated decades ago is unlikely to still count as renovated in 2014–15, and recent work is what's relevant to the client. Because renovated houses are few, combinations with at least 3 sales in each group are kept, so individual comparisons are noisy. The median across all comparisons is the more reliable figure.

With this approach, recently renovated older houses sell for a median of **about 10% more** than comparable unrenovated houses, and for more in 21 of 28 comparisons. H2 is therefore **supported**, with low confidence because of the small samples.

In [ ]:
# Recent renovations only
old["renovated_recent"] = old["yr_renovated"] >= 1990

# One label per zip code + size band combination
old["zip_band"] = old["zipcode"] + "_" + old["size_band"].astype(str)

# Median price and number of sales per combination, renovated vs not
medians = old.pivot_table(index="zip_band", columns="renovated_recent",
                          values="price", aggfunc="median")
counts = old.pivot_table(index="zip_band", columns="renovated_recent",
                         values="price", aggfunc="size", fill_value=0)
medians.columns = ["not_renovated", "renovated"]
counts.columns = ["n_not_renovated", "n_renovated"]

within = medians.join(counts)

# Keep combinations with at least 3 sales in both groups
within = within[(within["n_renovated"] >= 3) & (within["n_not_renovated"] >= 3)].copy()
within["premium_%"] = (within["renovated"] / within["not_renovated"] - 1) * 100

print(f"Comparisons: {len(within)}")
print(f"Median renovation premium: {within['premium_%'].median():.1f}%")
print(f"Renovated houses more expensive in {(within['premium_%'] > 0).sum()} of {len(within)}")

## 7. H3: Timing of sales

The data runs from May 2014 to May 2015, so sales are grouped by **year and month** to keep the two Mays separate. May 2015 is incomplete (the data ends on 27 May) and has fewer sales, so it's the least reliable month.

In [ ]:
# Group by year AND month so May 2014 and May 2015 stay separate
old["year_month"] = old["date"].dt.to_period("M")

monthly = old.groupby("year_month").agg(
    no_of_sales=("price", "size"),                        # how active the market is
    median_price_per_sqft=("price_per_sqft", "median"),   # price, adjusted for size
    median_price=("price", "median"),                     # actual sale price
)
monthly.round(0)

In [ ]:
labels = monthly.index.strftime("%b '%y")
ppsf_colour, price_colour = "cornflowerblue", "darkorange"

# Two stacked charts: prices on top (3 parts tall), sales below (2 parts tall)
fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(12, 9), sharex=True,
                                        gridspec_kw={"height_ratios": [3, 2]})

# Top chart, left axis: median price per sqft
line_ppsf, = ax_top.plot(labels, monthly["median_price_per_sqft"], marker="o",
                         color=ppsf_colour, linewidth=2,
                         label="Median price per sqft (left axis)")
ax_top.set_ylabel("Median price per sqft", color=ppsf_colour)
ax_top.tick_params(axis="y", colors=ppsf_colour)
ax_top.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax_top.grid(axis="y", alpha=0.3)

# Top chart, right axis: median sale price in $k
ax_price = ax_top.twinx()
line_price, = ax_price.plot(labels, monthly["median_price"] / 1000, marker="s",
                            color=price_colour, linewidth=2,
                            label="Median sale price (right axis)")
ax_price.set_ylabel("Median sale price", color=price_colour)
ax_price.tick_params(axis="y", colors=price_colour)
ax_price.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}k"))

# One legend for both lines, placed above the chart so it doesn't cover any points
ax_top.legend(handles=[line_ppsf, line_price], loc="lower center",
              bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)

# Bottom chart: number of sales, with the value on each bar
bars = ax_bottom.bar(labels, monthly["no_of_sales"], color="lightgrey")
ax_bottom.bar_label(bars, padding=2, fontsize=9)
ax_bottom.set_ylabel("Number of sales")
ax_bottom.set_ylim(0, monthly["no_of_sales"].max() * 1.15)   # room for the labels
ax_bottom.grid(axis="y", alpha=0.3)
ax_bottom.spines[["top", "right"]].set_visible(False)
ax_bottom.tick_params(axis="x", rotation=45)

fig.suptitle("Pre-1940 houses by month of sale", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

### 7.1 Seasonality or a rising market?

With only one year of data, seasonal effects can't be fully separated from general market growth. Comparing the same month a year apart removes the seasonal effect, so the remaining difference shows how much the market rose overall.

In [ ]:
may = old[old["date"].dt.month == 5]
may_compare = may.groupby(may["date"].dt.year)["price_per_sqft"].agg(["size", "median"])
display(may_compare.round(0))

growth = (may_compare["median"].iloc[1] / may_compare["median"].iloc[0] - 1) * 100
print(f"May 2014 → May 2015 change in median price per sqft: {growth:.1f}%")

**Sales volume** is clearly seasonal: spring and summer are much busier than winter, which has roughly half as many sales as summer. For a seller, more buyers means a better chance of a quick sale, which frees up capital for the next project.

**Prices** were highest in spring 2015, but older houses sold for about 11% more per sqft in May 2015 than in May 2014. Much of spring's price advantage may therefore be general market growth rather than seasonality. H3 is **partly supported**: volume is clearly seasonal, while the price effect is uncertain with one year of data.

The two lines in the price chart use different y-axes, so where they appear to cross has no meaning.

## 8. Conclusions

### Insights

1. **Geographical: location is the strongest driver of price per sqft.** Prices rise from south to north, and central and north Seattle are the most expensive areas. Older houses are concentrated in these same areas.
2. **Older houses are worth more per sqft, not less.** Within the same zip code, pre-1940 houses sell for a median of 8.8% more per sqft than newer houses. This contradicts the first part of H1.
3. **Prices are heavily right-skewed**, from \$78k to \$7.7M, so medians and price per sqft are used throughout.
4. **The market is seasonal in volume:** winter has about half as many sales as summer.

### Recommendations for Zachary

1. **Focus on eight zip codes** that rank in the top 10 for older houses on both price per sqft and total sale price: 98102, 98103, 98105, 98109, 98112, 98119, 98199 (central and north Seattle) and 98033 (Kirkland).
2. **Renovate when the cost is below about 10% of the expected sale price.** Recently renovated older houses sell for around 10% more than comparable unrenovated houses in the same area. This is based on a small sample, so treat it as a guide rather than a guarantee.
3. **List in spring or early summer** to reach the most buyers and sell faster. Prices were also highest in spring, but part of that reflects a rising market over the year.

### Limitations

- The data covers only one year (May 2014 – May 2015), so seasonal effects can't be fully separated from market trends.
- There are no purchase prices or renovation costs, so sale price is used as a proxy for returns.
- Renovation records are incomplete: some "unrenovated" houses may have been updated without it being recorded.
- Several comparisons rely on small samples, especially for renovated houses within zip codes.